# Cleaning the Gaia DR3 Pleiades sample

This notebook prepares the broad Gaia DR3 sample for clustering. We will inspect missing measurements and astrometric quality before deciding which rows are suitable for DBSCAN.

The raw catalogue is treated as read-only. Any filtered or derived table will be saved separately in `data/processed/`.

## 1. Imports and project paths

We use:

- `sys` to confirm which Python interpreter runs the notebook.
- `Path` to construct paths that work whether the notebook is launched from the project root or from `notebooks/`.
- NumPy for numerical checks, including finite-value masks.
- Astropy's `Table` class to read the FITS catalogue while preserving Gaia column metadata.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table

print(f"Python interpreter: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")

In [ ]:
project_root = Path.cwd()

if project_root.name.lower() == "notebooks":
    project_root = project_root.parent

raw_data_path = (
    project_root
    / "data"
    / "raw"
    / "gaia_dr3_pleiades_3deg_all_sources.fits"
)
processed_data_dir = project_root / "data" / "processed"
figures_dir = project_root / "figures"

processed_data_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")
print(f"Raw catalogue exists: {raw_data_path.exists()}")

## 2. Load the raw catalogue

`Table.read` loads the saved FITS file without changing it. The variable `gaia_raw` exists only in memory, so later experiments cannot alter the file unless we explicitly write to it.

In [ ]:
if not raw_data_path.exists():
    raise FileNotFoundError(
        "The combined raw catalogue is missing. Run Notebook 01 first."
    )

gaia_raw = Table.read(raw_data_path, format="fits")
raw_row_count = len(gaia_raw)

print(f"Rows: {len(gaia_raw):,}")
print(f"Columns: {len(gaia_raw.colnames)}")

## 3. Confirm the catalogue structure

Before cleaning, we check that the key measurements needed later are present. This is a structural validation step, not a membership selection.

In [ ]:
required_columns = [
    "source_id",
    "ra",
    "dec",
    "parallax",
    "parallax_error",
    "pmra",
    "pmra_error",
    "pmdec",
    "pmdec_error",
    "ruwe",
    "phot_g_mean_mag",
    "phot_bp_mean_mag",
    "phot_rp_mean_mag",
]

missing_columns = [
    name for name in required_columns if name not in gaia_raw.colnames
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("All required columns are present.")
gaia_raw[:5]

## 4. Inspect missing measurements

DBSCAN cannot use rows containing missing or non-finite values in its input features. Before removing anything, we count these values column by column.

Astropy may represent unavailable catalogue values either with a mask or as `NaN`. The helper function below checks for both cases. At this stage, a row is excluded only when a required numerical value is unavailable; we are not yet applying scientific quality thresholds or deciding cluster membership.

In [ ]:
def missing_value_mask(column):
    """Return True where an Astropy column is masked or non-finite."""
    masked = np.ma.getmaskarray(column)
    values = np.ma.filled(column, np.nan)
    return masked | ~np.isfinite(values)


columns_to_check = [
    "parallax",
    "parallax_error",
    "pmra",
    "pmra_error",
    "pmdec",
    "pmdec_error",
    "ruwe",
    "phot_g_mean_mag",
    "phot_bp_mean_mag",
    "phot_rp_mean_mag",
]

missing_summary = Table(
    names=("column", "missing_count", "missing_percent"),
    dtype=("U30", int, float),
)

for name in columns_to_check:
    missing_count = np.count_nonzero(missing_value_mask(gaia_raw[name]))
    missing_summary.add_row(
        (name, missing_count, 100 * missing_count / len(gaia_raw))
    )

missing_summary["missing_percent"].format = ".3f"
missing_summary

### Build the finite-astrometry mask

The later DBSCAN analysis will use parallax, `pmra`, and `pmdec` as its core feature space. A source must therefore have finite values for all three. Combining the three checks with `&` means that a row is retained only when every condition is true.

We store this Boolean mask without modifying `gaia_raw`. This lets us report exactly how many sources would be retained before creating a cleaned copy.

In [ ]:
clustering_features = ["parallax", "pmra", "pmdec"]

has_finite_astrometry = np.ones(len(gaia_raw), dtype=bool)

for name in clustering_features:
    has_finite_astrometry &= ~missing_value_mask(gaia_raw[name])

n_finite = np.count_nonzero(has_finite_astrometry)
n_missing = len(gaia_raw) - n_finite

print(f"Sources with finite parallax, pmra, and pmdec: {n_finite:,}")
print(f"Sources excluded for missing astrometry: {n_missing:,}")
print(f"Fraction retained: {n_finite / len(gaia_raw):.2%}")

## 5. Create a working table with complete astrometry

We now apply the objective finite-value requirement and immediately call `.copy()`. This creates an independent table for later cleaning steps, while `gaia_raw` remains the full downloaded catalogue.

We will not save this intermediate table yet because we still need to make and document the astrometric quality choices.

In [ ]:
gaia_finite = gaia_raw[has_finite_astrometry].copy()

print(f"Raw rows: {len(gaia_raw):,}")
print(f"Working rows: {len(gaia_finite):,}")

assert len(gaia_raw) == raw_row_count
assert all(
    np.all(~missing_value_mask(gaia_finite[name]))
    for name in clustering_features
)

## 6. Inspect astrometric quality indicators

A source can have finite measurements without those measurements being equally reliable. Before choosing filters, we examine several diagnostics:

- `ruwe` summarizes how well Gaia's single-source astrometric model fits the observations. Larger values can signal a poor fit, but there is no universal cutoff for every scientific problem.
- `visibility_periods_used` counts groups of observations separated in time. More periods generally provide a better-constrained astrometric solution.
- The three uncertainty columns describe the reported precision of the measurements that will enter DBSCAN. Their units match the associated measurements: milliarcseconds for parallax and milliarcseconds per year for proper motion.

The percentiles show the shape of each distribution without letting a few extreme values determine the scale. A percentile of 95 means that 95% of values lie at or below that number.

In [ ]:
quality_columns = [
    "ruwe",
    "visibility_periods_used",
    "parallax_error",
    "pmra_error",
    "pmdec_error",
]

quality_summary = Table(
    names=("column", "minimum", "median", "percentile_95", "maximum"),
    dtype=("U30", float, float, float, float),
)

for name in quality_columns:
    values = np.asarray(gaia_finite[name], dtype=float)
    minimum, median, percentile_95, maximum = np.percentile(
        values, [0, 50, 95, 100]
    )
    quality_summary.add_row(
        (name, minimum, median, percentile_95, maximum)
    )

for name in quality_summary.colnames[1:]:
    quality_summary[name].format = ".3f"

quality_summary

### Pause and interpret

Compare each median with its 95th percentile and maximum. Which diagnostic appears to have the strongest high-value tail? In the next section we will visualize the most relevant distributions and discuss defensible quality criteria before filtering them.

## 7. Visualize the quality distributions

Summary statistics tell us that high-value tails exist, but histograms show whether those tails are gradual or separated from the main population. We plot RUWE and `visibility_periods_used` because they describe complementary aspects of an astrometric solution: goodness of fit and observational time coverage.

For the RUWE panel, the horizontal range ends at the 99.5th percentile. This prevents a very small number of extreme values from compressing the main distribution. It does **not** remove those sources from `gaia_finite`. The dashed line at RUWE = 1.4 is shown as a commonly used reference point to investigate, not as an automatic or universal rule.

In [ ]:
ruwe_values = np.asarray(gaia_finite["ruwe"], dtype=float)
visibility_values = np.asarray(
    gaia_finite["visibility_periods_used"], dtype=int
)
ruwe_plot_max = np.percentile(ruwe_values, 99.5)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

axes[0].hist(
    ruwe_values[ruwe_values <= ruwe_plot_max],
    bins=60,
    color="tab:blue",
    alpha=0.8,
)
axes[0].axvline(1.4, color="tab:red", linestyle="--", label="RUWE = 1.4")
axes[0].set(
    xlabel="RUWE",
    ylabel="Number of sources",
    title="Astrometric goodness of fit",
)
axes[0].legend()

visibility_min = visibility_values.min()
visibility_max = visibility_values.max()
visibility_bins = np.arange(visibility_min - 0.5, visibility_max + 1.5)
axes[1].hist(
    visibility_values,
    bins=visibility_bins,
    color="tab:green",
    alpha=0.8,
)
axes[1].set(
    xlabel="Visibility periods used",
    ylabel="Number of sources",
    title="Observational time coverage",
)
axes[1].set_xticks(np.arange(visibility_min, visibility_max + 1))

quality_figure_path = figures_dir / "02_astrometric_quality_distributions.png"
fig.savefig(quality_figure_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"RUWE plotting limit (99.5th percentile): {ruwe_plot_max:.3f}")
print(f"Saved figure to: {quality_figure_path}")

### Questions for interpretation

1. Is the main RUWE distribution concentrated near 1, and what happens beyond the reference line?
2. Does `visibility_periods_used` have a long low-value tail, or is it confined to a fairly narrow range?
3. Based on these plots, which diagnostic seems more likely to affect the next cleaning decision?

## 8. Define and quantify baseline quality criteria

We will use two transparent baseline criteria:

- `ruwe < 1.4` to retain sources whose single-source astrometric fits are reasonably well behaved. This is a heuristic quality choice, not a universal physical boundary and not a membership cut. It may exclude genuine binaries, so we record its effect and can revisit it later.
- `visibility_periods_used >= 9` to require observations across at least nine separated time periods. In this catalogue, the plot suggests that every source with complete astrometry may already satisfy this condition; we verify that rather than assuming it.

The next cell counts the effect of each criterion separately before combining them.

In [ ]:
ruwe_limit = 1.4
minimum_visibility_periods = 9

passes_ruwe = np.asarray(gaia_finite["ruwe"]) < ruwe_limit
passes_visibility = (
    np.asarray(gaia_finite["visibility_periods_used"])
    >= minimum_visibility_periods
)

quality_effect = Table(
    rows=[
        ("Complete core astrometry", len(gaia_finite)),
        ("Passes RUWE criterion", np.count_nonzero(passes_ruwe)),
        ("Passes visibility criterion", np.count_nonzero(passes_visibility)),
        ("Passes both criteria", np.count_nonzero(passes_ruwe & passes_visibility)),
    ],
    names=("stage", "sources_retained"),
)
quality_effect["percent_of_finite_sample"] = (
    100 * quality_effect["sources_retained"] / len(gaia_finite)
)
quality_effect["percent_of_finite_sample"].format = ".2f"
quality_effect

### Apply the combined quality mask to a new copy

Now that its effect is explicit, we combine the criteria and create `gaia_clean`. The previous tables remain available: `gaia_raw` contains every downloaded row, and `gaia_finite` contains every row with the three clustering measurements.

In [ ]:
passes_astrometric_quality = passes_ruwe & passes_visibility
gaia_clean = gaia_finite[passes_astrometric_quality].copy()

n_quality_removed = len(gaia_finite) - len(gaia_clean)

print(f"Sources before quality criteria: {len(gaia_finite):,}")
print(f"Sources removed by quality criteria: {n_quality_removed:,}")
print(f"Sources in cleaned working table: {len(gaia_clean):,}")
print(f"Fraction of finite sample retained: {len(gaia_clean) / len(gaia_finite):.2%}")

### Interpretation

If the visibility-period criterion retains 100% of the finite-astrometry sample, it is a documented safeguard but does not drive the cleaning result. The change in sample size then comes entirely from RUWE. This distinction is important when we later test whether the membership result is sensitive to the adopted RUWE limit.

## 9. Inspect the cleaned astrometric feature space

Before scaling or clustering, we should see the measurements in their physical units. The parallax histogram shows the line-of-sight distance information, while the proper-motion plane shows how sources move across the sky. A real open cluster may appear as a localized overdensity because its members lie at similar distances and share a common space motion.

The plots use the 0.1st to 99.9th percentiles as display limits. This keeps a few extreme values from controlling the axes while retaining 99.8% of the cleaned sample. No rows are removed from `gaia_clean`. The proper-motion panel uses hexagonal bins rather than plotting every source individually; colour represents the number of sources in each bin on a logarithmic scale.

In [ ]:
parallax_values = np.asarray(gaia_clean["parallax"], dtype=float)
pmra_values = np.asarray(gaia_clean["pmra"], dtype=float)
pmdec_values = np.asarray(gaia_clean["pmdec"], dtype=float)

parallax_limits = np.percentile(parallax_values, [0.1, 99.9])
pmra_limits = np.percentile(pmra_values, [0.1, 99.9])
pmdec_limits = np.percentile(pmdec_values, [0.1, 99.9])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

axes[0].hist(
    parallax_values,
    bins=100,
    range=parallax_limits,
    color="tab:purple",
    alpha=0.8,
)
axes[0].set(
    xlabel="Parallax (mas)",
    ylabel="Number of sources",
    title="Cleaned parallax distribution",
)

proper_motion_density = axes[1].hexbin(
    pmra_values,
    pmdec_values,
    gridsize=100,
    extent=(*pmra_limits, *pmdec_limits),
    mincnt=1,
    bins="log",
    cmap="viridis",
)
axes[1].set(
    xlim=pmra_limits,
    ylim=pmdec_limits,
    xlabel=r"$\mu_{\alpha*}$ (mas yr$^{-1}$)",
    ylabel=r"$\mu_{\delta}$ (mas yr$^{-1}$)",
    title="Cleaned proper-motion density",
)
fig.colorbar(
    proper_motion_density,
    ax=axes[1],
    label="Sources per hexagonal bin",
)

astrometry_figure_path = figures_dir / "02_cleaned_astrometric_distributions.png"
fig.savefig(astrometry_figure_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {astrometry_figure_path}")

### Questions for interpretation

1. Does the parallax distribution look symmetric, or does it contain a long tail or secondary feature?
2. Where is the dominant field-star concentration in the proper-motion plane?
3. Can you identify a smaller, compact overdensity away from the central field population?

At this stage, any visible overdensity is only a candidate feature. We will later ask DBSCAN to identify structure in the combined three-dimensional space rather than selecting members by eye.

## 10. Validate the cleaned catalogue

Before writing a derived file, we perform explicit checks that match our documented cleaning rules. We require unique Gaia source identifiers, finite clustering features, RUWE below the adopted limit, and at least the adopted number of visibility periods. An `assert` stops execution if a condition is unexpectedly false.

We deliberately do **not** require complete BP and RP photometry here. Those measurements are needed for the later colour–magnitude diagram, but not for astrometric clustering. Removing them now could discard otherwise useful membership candidates.

In [ ]:
source_ids = np.asarray(gaia_clean["source_id"])

assert len(np.unique(source_ids)) == len(gaia_clean)
assert all(
    np.all(~missing_value_mask(gaia_clean[name]))
    for name in clustering_features
)
assert np.all(np.asarray(gaia_clean["ruwe"]) < ruwe_limit)
assert np.all(
    np.asarray(gaia_clean["visibility_periods_used"])
    >= minimum_visibility_periods
)

print("Validation passed: source IDs are unique.")
print("Validation passed: clustering features are finite.")
print("Validation passed: all documented quality criteria are satisfied.")

## 11. Save the processed catalogue

The cleaned table is a derived product, so it belongs in `data/processed/`. Writing with `overwrite=True` makes the notebook reproducible: rerunning it replaces the previous derived version while leaving the raw download untouched.

The FITS file retains the Gaia columns and their units. The notebook itself records the complete reasoning behind the cleaning choices.

In [ ]:
cleaned_data_path = (
    processed_data_dir
    / "gaia_dr3_pleiades_3deg_cleaned.fits"
)

gaia_clean.write(cleaned_data_path, format="fits", overwrite=True)

saved_table = Table.read(cleaned_data_path, format="fits")
assert len(saved_table) == len(gaia_clean)
assert saved_table.colnames == gaia_clean.colnames

print(f"Saved {len(saved_table):,} rows to:")
print(cleaned_data_path)

## 12. Cleaning summary

This notebook made two conceptually different changes:

1. **Objective numerical requirement:** removed rows without finite parallax, `pmra`, or `pmdec`, because DBSCAN cannot operate on missing feature values.
2. **Exploratory quality choice:** retained sources with RUWE below 1.4. The visibility-period safeguard removed no additional sources because all finite-astrometry rows already had at least nine periods.

No parallax or proper-motion membership window was applied. The surrounding field population remains in the processed catalogue so that clustering can be assessed against it. The next notebook will scale the three astrometric features and explore DBSCAN parameters.